In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_paul.xlsx"
dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_saqib.xlsx"

df = pd.read_excel(dataset_total_path)
df_val = pd.read_excel(dataset_val_path)

print(df.shape)
print(df_val.shape)

(24, 376)
(12, 376)


In [3]:
Class_A = 'LP'
Class_B = 'SP'

df = df[(df["Class"] == Class_A) | (df["Class"] == Class_B)]
df_val = df_val[(df_val["Class"] == Class_A) | (df_val["Class"] == Class_B)]

In [4]:
def encodage(df):
    code = {
    'LP' : 1,
    'SP' : 0,
    'LN' : 3,
    'SN' : 4
}
# Appliquer ce dictionnaire aux colonnes du dataset, avec la fonction map
    for col in df.select_dtypes('object'):
        df[col] = df[col].map(code)

    return df


def preprocessing(df):
    df = encodage(df)

    X = df.drop(['Class'], axis = 1)
    y = df['Class']

    # compter le nombre d'échantillons restants dans le dataset après avoir été inputé
    print(y.value_counts())

    return X, y

In [5]:
X_total, y_total = preprocessing(df)
X_val, y_val = preprocessing(df_val)

Class
1    6
0    6
Name: count, dtype: int64
Class
0    3
1    3
Name: count, dtype: int64


In [6]:
print("var before log2 transormation : " , X_total.var(axis=0).mean())
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X_total = log2_transformer.fit_transform(X_total)
X_val = log2_transformer.fit_transform(X_val)
print("var after log2 transormation : " , X_total.var(axis=0).mean())

var before log2 transormation :  7.684178425552448e+16
var after log2 transormation :  5.49605691519022


In [7]:
def evaluation(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)

    print(confusion_matrix(y_val, ypred))
    print(classification_report(y_val, ypred))

## Logistic regression

In [8]:
LR_L1 = make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
evaluation(LR_L1)

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



In [11]:
# Fit the model first
LR_L1.fit(X_total, y_total)

# Access the coefficients
coefs = LR_L1.named_steps['logisticregression'].coef_[0]

# Get associated feature names
feature_names = X_total.columns

# Create a DataFrame of coefficients
import pandas as pd
coef_df = pd.DataFrame({
    'metabolite': feature_names,
    'coefficient': coefs
})

# Optional: filter non-zero coefficients (selected by L1)
selected = coef_df[coef_df['coefficient'] != 0]

selected.to_excel("C:/Users/tamer/Documents/PhD/ML/ML_coefs/LogReg_Lasso.xlsx", index=False)

## SVC

In [10]:
SVM_linear_L1 = make_pipeline(StandardScaler(), LinearSVC(penalty = 'l1', random_state = 0))
evaluation(SVM_linear_L1)

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



C:\Users\tamer\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
